In [70]:
import numpy as np
import pandas as pd
from sklearn.impute import KNNImputer
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from tensorflow.keras.callbacks import EarlyStopping

In [71]:
data=pd.read_csv('/kaggle/input/datasets/emadhamdy11111/berest-cancer-data/breast_cancer_dirty.csv')

data.head()

,id,Diagnosis,Radius_Mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,SMOOTHNESS_WORST,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,902727,B,13.28,13.72,85.79,541.8,0.08363,0.08575,0.05077,0.02864,...,17.37,96.59,623.7,0.1166,0.2685,0.2866,0.09173,0.2736,0.07320,NaN
1,8711002,B,13.15,15.34,85.31,538.9,0.09384,0.08498,0.09293,0.03483,...,20.50,97.67,677.3,0.1478,0.2256,0.3009,0.09722,0.3849,0.08633,NaN
2,9012568,b,15.19,13.21,97.65,711.8,0.07963,0.06934,NaN,0.02657,...,15.73,104.50,819.1,0.1126,0.1737,0.1362,0.08178,0.2487,0.06766,NaN
3,864018,b,11.34,21.26,72.48,396.5,0.08759,0.06575,0.05133,0.01899,...,29.15,83.99,518.1,0.1699,0.2196,0.3120,0.08278,0.2829,0.08832,NaN
4,892604,B,12.46,19.89,80.43,471.3,0.08451,0.10140,0.06830,0.03099,...,23.07,88.13,551.3,0.1050,0.2158,0.1904,0.07625,0.2685,0.07764,NaN


In [72]:
data.shape

(587, 33)

In [73]:
data.isna().sum()

id                           0
Diagnosis                    4
Radius_Mean                  0
 texture_mean               12
perimeter_mean               7
area_mean                    0
smoothness_mean              9
compactness_mean             0
concavity_mean              10
concave points_mean          0
symmetry_mean                0
fractal_dimension_mean       0
radius_se                    0
texture_se                   0
perimeter_se                 0
area_se                      0
smoothness_se                0
compactness_se               0
concavity_se                 0
concave points_se            0
symmetry_se                  0
fractal_dimension_se         0
radius_worst                 0
texture_worst                0
perimeter_worst              0
area_worst                   0
SMOOTHNESS_WORST             0
compactness_worst            0
concavity_worst              0
concave points_worst         0
symmetry_worst              15
fractal_dimension_worst      0
Unnamed:

In [74]:
data.dtypes

id                           int64
Diagnosis                   object
Radius_Mean                float64
 texture_mean              float64
perimeter_mean             float64
area_mean                   object
smoothness_mean            float64
compactness_mean           float64
concavity_mean             float64
concave points_mean        float64
symmetry_mean              float64
fractal_dimension_mean     float64
radius_se                  float64
texture_se                 float64
perimeter_se               float64
area_se                    float64
smoothness_se              float64
compactness_se             float64
concavity_se               float64
concave points_se          float64
symmetry_se                float64
fractal_dimension_se       float64
radius_worst               float64
texture_worst              float64
perimeter_worst            float64
area_worst                 float64
SMOOTHNESS_WORST           float64
compactness_worst          float64
concavity_worst     

In [75]:
data.duplicated().sum()

np.int64(15)

In [76]:
data=data.drop(columns='id')

In [77]:
data=data.drop(columns='Unnamed: 32')

In [78]:
data.head()

,Diagnosis,Radius_Mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,SMOOTHNESS_WORST,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,B,13.28,13.72,85.79,541.8,0.08363,0.08575,0.05077,0.02864,0.1617,...,14.24,17.37,96.59,623.7,0.1166,0.2685,0.2866,0.09173,0.2736,0.07320
1,B,13.15,15.34,85.31,538.9,0.09384,0.08498,0.09293,0.03483,0.1822,...,14.77,20.50,97.67,677.3,0.1478,0.2256,0.3009,0.09722,0.3849,0.08633
2,b,15.19,13.21,97.65,711.8,0.07963,0.06934,NaN,0.02657,0.1721,...,16.20,15.73,104.50,819.1,0.1126,0.1737,0.1362,0.08178,0.2487,0.06766
3,b,11.34,21.26,72.48,396.5,0.08759,0.06575,0.05133,0.01899,0.1487,...,13.01,29.15,83.99,518.1,0.1699,0.2196,0.3120,0.08278,0.2829,0.08832
4,B,12.46,19.89,80.43,471.3,0.08451,0.10140,0.06830,0.03099,0.1781,...,13.46,23.07,88.13,551.3,0.1050,0.2158,0.1904,0.07625,0.2685,0.07764


**drop duplicates**

In [79]:
data=data.drop_duplicates().reset_index(drop=True)

In [80]:
data['Diagnosis']=data['Diagnosis'].astype(str).str.strip()
data['Diagnosis']=data['Diagnosis'].astype('category')


In [81]:
data.columns = data.columns.str.strip()

In [82]:
data['area_mean']=data['area_mean'].astype(str).str.extract(r'(-?\d+\.?\d*)')

data['area_mean']=pd.to_numeric(data['area_mean'],errors='coerce')



In [83]:
print(data['Diagnosis'].unique())

['B', 'b', 'benign', 'M', 'Benign', 'Malignant', 'malignant', 'm', 'nan']
Categories (9, object): ['B', 'Benign', 'M', 'Malignant', ..., 'benign', 'm', 'malignant', 'nan']


In [84]:
data['Diagnosis'] = data['Diagnosis'].astype(str).str.strip().str.lower()

data['Diagnosis'] = data['Diagnosis'].replace({
    'b': 0,
    'benign': 0,
    'm': 1,
    'malignant': 1,
    'nan': float(
        'nan'
    )
})

print(data['Diagnosis'].value_counts(dropna=False))

Diagnosis
0.0    357
1.0    211
NaN      4
Name: count, dtype: int64


/tmp/ipykernel_58/3313525475.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data['Diagnosis'] = data['Diagnosis'].replace({


In [85]:
for col in data.columns:
    print(data[col].unique())

[ 0.  1. nan]
[  13.28    13.15    15.19    11.34    12.46    11.43    11.49     9.738
   16.17    16.35    15.13     9.683   19.53    16.27    12.43    13.68
   12.72  1427.      11.37    12.81    11.81    12.34    12.25    12.07
   15.46    12.87    18.77    10.17    11.8      9.787   15.66    11.76
   13.87    15.85    12.8     11.42    20.16     7.691   12.16    13.65
   14.06    11.66    10.2     12.32    18.94    12.23    11.14    13.75
   11.93     7.76     6.981   17.35    11.25    18.82    14.74    16.46
   18.63    14.42    11.52    18.22    13.66    12.27    12.04    18.49
   11.27    13.64    13.5     12.99    12.1     11.62    11.04    19.19
   13.94    11.71     9.742 1286.      16.3     13.46    14.92    11.54
   12.47    10.16    16.03     9.72    11.13     8.196   17.42    11.51
    9.668   16.6     15.49    25.22    11.29    21.16    11.22    19.4
   14.99    21.75    11.67    18.08    16.02    20.18    14.69    10.48
   17.02    15.34    17.14    11.68    12.75    10

In [86]:
mode_diagnosis=data['Diagnosis'].mode()[0]
data['Diagnosis']=data['Diagnosis'].fillna(mode_diagnosis)
data['Diagnosis'].isna().sum()

np.int64(0)

In [87]:
num_cols=data.select_dtypes(include=['float64','float32','int32','int64']).columns

for col in num_cols:
    q1=data[col].quantile(0.25)
    q3=data[col].quantile(0.75)
    iqr=q3-q1
    upper_bound=q3+(iqr*1.5)
    lower_bound=q1-(iqr*1.5)
    data[col]=np.clip(data[col],lower_bound,upper_bound)

imputer=KNNImputer(n_neighbors=5)

if(len(num_cols))>0:
    data[num_cols]=imputer.fit_transform(data[num_cols])

In [88]:
data.isna().sum()

Diagnosis                  0
Radius_Mean                0
texture_mean               0
perimeter_mean             0
area_mean                  0
smoothness_mean            0
compactness_mean           0
concavity_mean             0
concave points_mean        0
symmetry_mean              0
fractal_dimension_mean     0
radius_se                  0
texture_se                 0
perimeter_se               0
area_se                    0
smoothness_se              0
compactness_se             0
concavity_se               0
concave points_se          0
symmetry_se                0
fractal_dimension_se       0
radius_worst               0
texture_worst              0
perimeter_worst            0
area_worst                 0
SMOOTHNESS_WORST           0
compactness_worst          0
concavity_worst            0
concave points_worst       0
symmetry_worst             0
fractal_dimension_worst    0
dtype: int64

In [89]:
data.dtypes

Diagnosis                  float64
Radius_Mean                float64
texture_mean               float64
perimeter_mean             float64
area_mean                  float64
smoothness_mean            float64
compactness_mean           float64
concavity_mean             float64
concave points_mean        float64
symmetry_mean              float64
fractal_dimension_mean     float64
radius_se                  float64
texture_se                 float64
perimeter_se               float64
area_se                    float64
smoothness_se              float64
compactness_se             float64
concavity_se               float64
concave points_se          float64
symmetry_se                float64
fractal_dimension_se       float64
radius_worst               float64
texture_worst              float64
perimeter_worst            float64
area_worst                 float64
SMOOTHNESS_WORST           float64
compactness_worst          float64
concavity_worst            float64
concave points_worst

In [90]:
data.duplicated().sum()

np.int64(3)

In [91]:
data=data.drop_duplicates().reset_index(drop=True)

In [92]:
data.duplicated().sum()

np.int64(0)

In [93]:
X=data.drop(columns='Diagnosis')

y=data['Diagnosis']

In [94]:
x_train_val,x_test,y_train_val,y_test=train_test_split(X,y,stratify=y,test_size=0.2,random_state=42)

In [95]:
x_train,x_val,y_train,y_val=train_test_split(x_train_val,y_train_val,stratify=y_train_val,random_state=42,test_size=0.25)

In [102]:
scaler=StandardScaler()

x_train_scaled=scaler.fit_transform(x_train)
x_val_scaled=scaler.transform(x_val)
x_test_scaled=scaler.transform(x_test)

**Buld the model**

In [97]:
model=tf.keras.Sequential([
    layers.Input(shape=(30,)),
    layers.Dense(32,activation='relu'),
    layers.Dense(13,activation='relu'),
    layers.Dense(1,activation='sigmoid')
])

In [98]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['recall']
)

In [99]:
early_stop=EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

In [103]:
history=model.fit(
    x_train_scaled,
    y_train,
    epochs=100,
    validation_data=(x_val_scaled,y_val),
    verbose=1,
    batch_size=32,
    callbacks=[early_stop]
)

Epoch 1/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 2s 30ms/step - loss: 0.6087 - recall: 0.5280 - val_loss: 0.4700 - val_recall: 0.8095
Epoch 2/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.4485 - recall: 0.7840 - val_loss: 0.3466 - val_recall: 0.9048
Epoch 3/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.3480 - recall: 0.8480 - val_loss: 0.2627 - val_recall: 0.9286
Epoch 4/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.2765 - recall: 0.8720 - val_loss: 0.2124 - val_recall: 0.9286
Epoch 5/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.2322 - recall: 0.8800 - val_loss: 0.1800 - val_recall: 0.9286
Epoch 6/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.2021 - recall: 0.9040 - val_loss: 0.1610 - val_recall: 0.9524
Epoch 7/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.1815 - recall: 0.9040 - val_loss: 0.1483 - val_recall: 0.9524
Epoch 8/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.1672 - recall: 0.9200 - val_loss: 0.1396 - val_recall: 0.9524
Epoch 9/10

In [104]:
test_loss, test_recall = model.evaluate(x_test_scaled, y_test)

print(f'Test Loss: {test_loss:.4f}')
print(f'Test Recall: {test_recall:.4f}')

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0680 - recall: 0.9762
Test Loss: 0.0680
Test Recall: 0.9762


In [105]:
model.save('breast_cancer_model.keras')